# Build `RM_SDSS_df.pkl` with explicit central/member naming

This notebook rebuilds the redMaPPer cluster-member table used by the projected matching workflow.  The science-facing convention is:

- `_central` columns are cluster-center/BCG quantities.  For clusters with an identified BGS central galaxy, `RA_central`, `DEC_central`, and `Z_SPEC_central` are patched from `RA_BGS`, `DEC_BGS`, and `Z_BGS`.
- `_member` columns are redMaPPer member-galaxy quantities.

For backward compatibility with older notebooks, the output also writes legacy aliases such as `RA_x`, `DEC_x`, `Z_SPEC_x`, `RA_y`, `DEC_y`, and `Z_SPEC_y`.  New code should prefer the `_central` and `_member` names.

The output is written to `catalogs/RM_SDSS_df.pkl`.


In [ ]:
from pathlib import Path
import sys
import pickle

import numpy as np
pandas = None
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import astropy.units as u
import astropy.cosmology.units as cu
from astropy.cosmology import Planck18
from astropy.coordinates import SkyCoord
from astropy.table import Table

# In NERSC notebooks, keep the child repo importable.
REPO_DIR = Path('/global/homes/z/zzhang13/DESI/Projection')
if not REPO_DIR.exists():
    REPO_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == 'make_catalogs' else Path.cwd().resolve()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

CATALOG_DIR = REPO_DIR / 'catalogs'
CATALOG_DIR.mkdir(parents=True, exist_ok=True)

# Raw redMaPPer files are often in the older DESI/catalogs folder on NERSC.
RAW_CATALOG_DIRS = [
    CATALOG_DIR,
    Path('/global/homes/z/zzhang13/DESI/catalogs'),
    Path('/global/cfs/cdirs/desi/users/zzhang13/catalogs'),
    Path.cwd(),
]

RM_CLUSTER_BASENAME = 'redmapper_dr8_public_v6.3_catalog.fits'
RM_MEMBER_BASENAME = 'redmapper_dr8_public_v6.3_members.fits'
BGS_CENTER_CATALOG = CATALOG_DIR / 'bgs_clus_RM_gal_matched.pickle'
OUTPUT_PICKLE = CATALOG_DIR / 'RM_SDSS_df.pkl'
PLOT_DIR = REPO_DIR / 'make_catalogs' / 'diagnostic_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

COSMO = Planck18
H = COSMO.H0.value / 100.0
Z_BINS = [(0.1, 0.2), (0.2, 0.3), (0.3, 0.4)]

print('REPO_DIR:', REPO_DIR)
print('CATALOG_DIR:', CATALOG_DIR)
print('BGS center catalog:', BGS_CENTER_CATALOG)
print('Output:', OUTPUT_PICKLE)


In [ ]:
def first_existing(basename, search_dirs):
    for directory in search_dirs:
        path = Path(directory) / basename
        if path.exists():
            return path
    searched = '\n  '.join(str(Path(d) / basename) for d in search_dirs)
    raise FileNotFoundError(f'Could not find {basename}. Searched:\n  {searched}')


def fits_to_scalar_dataframe(path):
    """Read a FITS table and keep columns that pandas can store cleanly."""
    tab = Table.read(path)
    keep = []
    for col in tab.colnames:
        arr = np.asarray(tab[col])
        if arr.ndim <= 1:
            keep.append(col)
    return tab[keep].to_pandas()


def read_pickle_table(path):
    with Path(path).open('rb') as handle:
        obj = pickle.load(handle)
    if isinstance(obj, Table):
        return obj
    if isinstance(obj, pd.DataFrame):
        return Table.from_pandas(obj)
    if hasattr(obj, 'to_pandas'):
        return Table.from_pandas(obj.to_pandas())
    return Table(obj)


def projected_radius_hmpc(ra_center, dec_center, z_center, ra_member, dec_member):
    """Projected comoving separation in h^-1 Mpc at the cluster redshift."""
    center = SkyCoord(ra=np.asarray(ra_center, dtype=float) * u.deg,
                      dec=np.asarray(dec_center, dtype=float) * u.deg)
    member = SkyCoord(ra=np.asarray(ra_member, dtype=float) * u.deg,
                      dec=np.asarray(dec_member, dtype=float) * u.deg)
    theta = center.separation(member).rad
    dm = (np.asarray(z_center, dtype=float) * cu.redshift).to(
        u.Mpc, cu.redshift_distance(COSMO, kind='comoving')
    ).value
    return theta * dm * H


def choose_bgs_cluster_centers(path):
    """Return one BGS central row per redMaPPer ID.

    The central source table is galaxy-level.  We keep rows with central_flag,
    then choose the row closest to the original RM center if duplicate central
    rows exist for the same ID.
    """
    tab = read_pickle_table(path)
    required = ['ID', 'RA_BGS', 'DEC_BGS', 'Z_BGS', 'RA_x', 'DEC_x', 'central_flag']
    missing = [c for c in required if c not in tab.colnames]
    if missing:
        raise KeyError(f'{path} is missing required columns: {missing}')

    central = tab[np.asarray(tab['central_flag'], dtype=bool)]
    if len(central) == 0:
        return pd.DataFrame(columns=['ID', 'RA_BGS_CENTER', 'DEC_BGS_CENTER', 'Z_BGS_CENTER'])

    sep = SkyCoord(central['RA_BGS'] * u.deg, central['DEC_BGS'] * u.deg).separation(
        SkyCoord(central['RA_x'] * u.deg, central['DEC_x'] * u.deg)
    ).arcsec
    df = central[['ID', 'RA_BGS', 'DEC_BGS', 'Z_BGS']].to_pandas()
    df['center_sep_arcsec'] = sep
    df = df.sort_values(['ID', 'center_sep_arcsec']).drop_duplicates('ID', keep='first')
    return df.rename(columns={
        'RA_BGS': 'RA_BGS_CENTER',
        'DEC_BGS': 'DEC_BGS_CENTER',
        'Z_BGS': 'Z_BGS_CENTER',
    })


In [ ]:
rm_cluster_path = first_existing(RM_CLUSTER_BASENAME, RAW_CATALOG_DIRS)
rm_member_path = first_existing(RM_MEMBER_BASENAME, RAW_CATALOG_DIRS)

print('Reading cluster file:', rm_cluster_path)
print('Reading member file :', rm_member_path)

rm_clusters = fits_to_scalar_dataframe(rm_cluster_path)
rm_members = fits_to_scalar_dataframe(rm_member_path)

print('RM clusters:', rm_clusters.shape)
print('RM members :', rm_members.shape)
print('Cluster columns sample:', list(rm_clusters.columns[:15]))
print('Member columns sample :', list(rm_members.columns[:15]))


In [ ]:
# Use human-readable suffixes rather than pandas' default _x/_y convention.
# _central = cluster/BCG quantity; _member = redMaPPer member-galaxy quantity.
rm_sdss = pd.merge(
    rm_clusters,
    rm_members,
    on='ID',
    how='inner',
    suffixes=('_central', '_member'),
)

# Some member-only columns do not receive a suffix during the merge.
# Rename them so their role is still explicit.
member_only_renames = {
    'R': 'R_member',
    'P': 'P_member',
    'P_FREE': 'P_FREE_member',
    'THETA_I': 'THETA_I_member',
    'THETA_R': 'THETA_R_member',
}
rm_sdss = rm_sdss.rename(columns={k: v for k, v in member_only_renames.items() if k in rm_sdss.columns})

# redMaPPer radial scale used by the old notebook.
if 'R_LAMBDA' not in rm_sdss.columns:
    rm_sdss['R_LAMBDA'] = (rm_sdss['LAMBDA'] / 100.0) ** 0.2

# Keep a permanent record of the original redMaPPer/SDSS cluster-center columns
# before the optional BGS-central patch.
for col in ['RA_central', 'DEC_central', 'Z_SPEC_central']:
    if col in rm_sdss.columns and f'{col}_RM_ORIG' not in rm_sdss.columns:
        rm_sdss[f'{col}_RM_ORIG'] = rm_sdss[col]

print('Merged RM cluster/member rows:', rm_sdss.shape)
print([c for c in [
    'ID', 'RA_central', 'DEC_central', 'Z_SPEC_central',
    'RA_member', 'DEC_member', 'Z_SPEC_member', 'R_member', 'P_member', 'R_LAMBDA'
] if c in rm_sdss.columns])


In [ ]:
centers = choose_bgs_cluster_centers(BGS_CENTER_CATALOG)
print('BGS central IDs available:', len(centers))

rm_sdss = rm_sdss.merge(centers, on='ID', how='left')
has_bgs_center = (
    np.isfinite(rm_sdss['RA_BGS_CENTER'])
    & np.isfinite(rm_sdss['DEC_BGS_CENTER'])
    & np.isfinite(rm_sdss['Z_BGS_CENTER'])
)

rm_sdss['CENTER_SOURCE'] = np.where(has_bgs_center, 'BGS_SDSS_central', 'redMaPPer_SDSS_original')
rm_sdss.loc[has_bgs_center, 'RA_central'] = rm_sdss.loc[has_bgs_center, 'RA_BGS_CENTER']
rm_sdss.loc[has_bgs_center, 'DEC_central'] = rm_sdss.loc[has_bgs_center, 'DEC_BGS_CENTER']
rm_sdss.loc[has_bgs_center, 'Z_SPEC_central'] = rm_sdss.loc[has_bgs_center, 'Z_BGS_CENTER']

# Remove member rows that correspond to the central itself when both OBJIDs are present.
mask = np.isfinite(rm_sdss['Z_SPEC_central']) & (rm_sdss['Z_SPEC_central'] > 0)
if {'OBJID_central', 'OBJID_member'}.issubset(rm_sdss.columns):
    mask &= rm_sdss['OBJID_central'] != rm_sdss['OBJID_member']
if 'Z_SPEC_member' in rm_sdss.columns:
    # Do not repeat/count the central galaxy as a member row.  This mirrors
    # the original notebook cut:
    # merge_df = merge_df.iloc[np.where(merge_df['Z_SPEC_y'] != merge_df['Z_SPEC_x'])]
    mask &= rm_sdss['Z_SPEC_member'] != rm_sdss['Z_SPEC_central']

rm_sdss = rm_sdss.loc[mask].reset_index(drop=True)

# Backward-compatible aliases for older downstream scripts.
legacy_aliases = {
    'RA_x': 'RA_central',
    'DEC_x': 'DEC_central',
    'Z_SPEC_x': 'Z_SPEC_central',
    'OBJID_x': 'OBJID_central',
    'MODEL_MAG_R_x': 'MODEL_MAG_R_central',
    'MODEL_MAGERR_R_x': 'MODEL_MAGERR_R_central',
    'RA_y': 'RA_member',
    'DEC_y': 'DEC_member',
    'Z_SPEC_y': 'Z_SPEC_member',
    'OBJID_y': 'OBJID_member',
    'MODEL_MAG_R_y': 'MODEL_MAG_R_member',
    'MODEL_MAGERR_R_y': 'MODEL_MAGERR_R_member',
    'R': 'R_member',
    'P': 'P_member',
}
for legacy_name, preferred_name in legacy_aliases.items():
    if preferred_name in rm_sdss.columns:
        rm_sdss[legacy_name] = rm_sdss[preferred_name]

print('Rows after central/self and good-z cuts:', rm_sdss.shape)
print(rm_sdss['CENTER_SOURCE'].value_counts(dropna=False))


In [ ]:
rm_sdss['R_2D_HMPC_FROM_CENTRAL_MEMBER'] = projected_radius_hmpc(
    rm_sdss['RA_central'],
    rm_sdss['DEC_central'],
    rm_sdss['Z_SPEC_central'],
    rm_sdss['RA_member'],
    rm_sdss['DEC_member'],
)
rm_sdss['R_OVER_RLAMBDA_FROM_CENTRAL_MEMBER'] = rm_sdss['R_2D_HMPC_FROM_CENTRAL_MEMBER'] / rm_sdss['R_LAMBDA']
rm_sdss['DZ_MEMBER_CLUSTER'] = (rm_sdss['Z_SPEC_member'] - rm_sdss['Z_SPEC_central']) / (1.0 + rm_sdss['Z_SPEC_central'])

# Legacy aliases for previous diagnostic naming.
rm_sdss['R_2D_HMPC_FROM_XY'] = rm_sdss['R_2D_HMPC_FROM_CENTRAL_MEMBER']
rm_sdss['R_OVER_RLAMBDA_FROM_XY'] = rm_sdss['R_OVER_RLAMBDA_FROM_CENTRAL_MEMBER']

summary = rm_sdss[[
    'R_member',
    'R_2D_HMPC_FROM_CENTRAL_MEMBER',
    'R_OVER_RLAMBDA_FROM_CENTRAL_MEMBER',
    'P_member',
    'DZ_MEMBER_CLUSTER',
]].describe(percentiles=[0.01, 0.16, 0.5, 0.84, 0.99])
display(summary)


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.3))
finite = np.isfinite(rm_sdss['R_member']) & np.isfinite(rm_sdss['R_2D_HMPC_FROM_CENTRAL_MEMBER'])
ax.hexbin(
    rm_sdss.loc[finite, 'R_member'],
    rm_sdss.loc[finite, 'R_2D_HMPC_FROM_CENTRAL_MEMBER'],
    gridsize=80, bins='log', mincnt=1, cmap='mako' if 'mako' in plt.colormaps() else 'viridis'
)
lim = np.nanpercentile(np.r_[
    rm_sdss.loc[finite, 'R_member'],
    rm_sdss.loc[finite, 'R_2D_HMPC_FROM_CENTRAL_MEMBER'],
], 99.5)
ax.plot([0, lim], [0, lim], color='black', lw=1.2, ls='--', label='1:1')
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel(r'redMaPPer member $R$ [$h^{-1}{\rm Mpc}$]')
ax.set_ylabel(r'recomputed $R_{\perp}$ from central/member positions [$h^{-1}{\rm Mpc}$]')
ax.legend(frameon=False, loc='upper left')
ax.tick_params(direction='in', top=True, right=True)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'RM_SDSS_member_radius_consistency.png', dpi=180)
plt.show()


In [ ]:
fig, axs = plt.subplots(1, len(Z_BINS), figsize=(14.5, 4.2), sharey=True, constrained_layout=True)
last = None
for ax, (zlo, zhi) in zip(axs, Z_BINS):
    sub = rm_sdss[(rm_sdss['Z_SPEC_central'] >= zlo) & (rm_sdss['Z_SPEC_central'] < zhi)]
    ok = (
        np.isfinite(sub['R_OVER_RLAMBDA_FROM_CENTRAL_MEMBER'])
        & np.isfinite(sub['P_member'])
        & (sub['R_OVER_RLAMBDA_FROM_CENTRAL_MEMBER'] > 0)
    )
    last = ax.hist2d(
        sub.loc[ok, 'R_OVER_RLAMBDA_FROM_CENTRAL_MEMBER'],
        sub.loc[ok, 'P_member'],
        bins=(70, 55),
        range=[[0, 3.0], [0, 1.0]],
        norm=LogNorm(),
        cmap='viridis',
    )
    ax.set_xlabel(r'$R_\perp/R_\lambda$')
    ax.text(0.05, 0.93, rf'${zlo}<z_{{\rm spec,BCG}}<{zhi}$' + '\n' + rf'$N={len(sub):,}$',
            transform=ax.transAxes, ha='left', va='top',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.82))
    ax.tick_params(direction='in', top=True, right=True)
axs[0].set_ylabel(r'redMaPPer membership probability $P_{\rm mem}$')
fig.colorbar(last[3], ax=axs, label='member count')
fig.savefig(PLOT_DIR / 'RM_SDSS_pmem_vs_radius_by_z.png', dpi=180)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.8))
for source, color in [('BGS_SDSS_central', 'tab:blue'), ('redMaPPer_SDSS_original', '0.45')]:
    sub = rm_sdss[rm_sdss['CENTER_SOURCE'] == source].drop_duplicates('ID')
    ax.hist(sub['Z_SPEC_central'], bins=np.linspace(0, 0.45, 46), histtype='step', lw=1.8, color=color, label=f'{source} ({len(sub):,} clusters)')
ax.set_xlabel(r'$z_{\rm spec,BCG}$ from `Z_SPEC_central`')
ax.set_ylabel('cluster count')
ax.legend(frameon=False)
ax.tick_params(direction='in', top=True, right=True)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'RM_SDSS_center_source_redshift_distribution.png', dpi=180)
plt.show()


In [ ]:
# A visual spot check: choose a few clusters with many high-probability members.
cluster_counts = (
    rm_sdss[rm_sdss['P_member'] > 0.2]
    .groupby('ID')
    .size()
    .sort_values(ascending=False)
)
example_ids = list(cluster_counts.head(6).index)

fig, axs = plt.subplots(2, 3, figsize=(12, 7.4), constrained_layout=True)
for ax, cid in zip(axs.flat, example_ids):
    sub = rm_sdss[rm_sdss['ID'] == cid].copy()
    cx, cy = float(sub['RA_central'].iloc[0]), float(sub['DEC_central'].iloc[0])
    dx = (sub['RA_member'] - cx) * np.cos(np.deg2rad(cy)) * 60.0
    dy = (sub['DEC_member'] - cy) * 60.0
    sc = ax.scatter(dx, dy, c=sub['P_member'], s=18, cmap='viridis', vmin=0, vmax=1, edgecolor='none')
    ax.scatter([0], [0], marker='+', s=120, color='crimson', lw=2.0, label='cluster center')
    ax.axhline(0, color='0.85', lw=0.8)
    ax.axvline(0, color='0.85', lw=0.8)
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_title(rf'ID {cid}, $z={sub["Z_SPEC_central"].iloc[0]:.3f}$', fontsize=10)
    ax.set_xlabel(r'$\Delta{\rm RA}\cos\delta$ [arcmin]')
    ax.set_ylabel(r'$\Delta{\rm Dec}$ [arcmin]')
    ax.tick_params(direction='in', top=True, right=True)
fig.colorbar(sc, ax=axs, label=r'$P_{\rm mem}$')
fig.savefig(PLOT_DIR / 'RM_SDSS_member_sky_spotcheck.png', dpi=180)
plt.show()


In [ ]:
with OUTPUT_PICKLE.open('wb') as handle:
    pickle.dump(rm_sdss, handle, protocol=pickle.HIGHEST_PROTOCOL)

print('Saved:', OUTPUT_PICKLE)
print('Rows:', len(rm_sdss))
print('Unique clusters:', rm_sdss['ID'].nunique())
print('Diagnostic plots saved in:', PLOT_DIR)
